# 18b — Analytics Brand Judge Context Fix

این notebook فقط دو case برند را دوباره **judge** می‌کند.

Manager Answer دوباره اجرا نمی‌شود. Context عددی و `top_brands` به‌صورت
deterministic از Python بازسازی می‌شود و فقط دو Judge call جدید داریم.

علت rerun: evaluator اولیه هنگام compact کردن result، جدول‌های supporting
مثل `top_brands` را قبل از Judge حذف می‌کرد؛ بنابراین Judge نمی‌توانست
ادعاهای برند را با context واقعی پاسخ verify کند.


In [1]:
from pathlib import Path
import os
import sys

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

load_dotenv(
    PROJECT_ROOT
    / ".env"
)

from src.rag.config import load_config
from src.rag.evaluation import (
    AnalyticsEvaluationDataset,
    ManagerAnalyticsEvaluator,
    analytics_failure_summary,
)
from src.rag.evaluation.analytics_metrics import (
    summarize_analytics_results,
)
from src.rag.evaluation.analytics_runtime import (
    load_analytics_evaluation_context,
)

In [2]:
evaluation_config = load_config(
    PROJECT_ROOT
    / "configs"
    / "analytics_evaluation.yaml"
)[
    "analytics_evaluation"
]

dataset = (
    AnalyticsEvaluationDataset
    .load(
        PROJECT_ROOT
        / evaluation_config[
            "dataset"
        ][
            "path"
        ]
    )
)

context = (
    load_analytics_evaluation_context(
        project_root=(
            PROJECT_ROOT
        ),
        api_key=os.environ[
            "METIS_API_KEY"
        ],
        base_url=os.environ[
            "METIS_BASE_URL"
        ],
    )
)

evaluator = (
    ManagerAnalyticsEvaluator(
        analytics_pipeline=(
            context.pipeline
        ),
        judge=context.judge,
        dataset=dataset,
        generic_brand_values=(
            context.analytics_config[
                "audit"
            ][
                "generic_brand_values"
            ]
        ),
        rating_max=(
            context.analytics_config[
                "aggregation"
            ][
                "product_rating_max"
            ]
        ),
        weights=(
            evaluation_config[
                "weights"
            ]
        ),
    )
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / evaluation_config[
        "output"
    ][
        "directory"
    ]
)

CHECKPOINT_PATH = (
    OUTPUT_DIR
    / evaluation_config[
        "output"
    ][
        "checkpoint"
    ]
)

print(
    "Checkpoint:",
    CHECKPOINT_PATH,
)

Checkpoint: /home/ali/Desktop/projects/digikala-ai-assistant/data/evaluation/analytics/analytics_checkpoint.jsonl


In [3]:
BRAND_CASES = [
    "a004",
    "a012",
]

rejudged = (
    evaluator
    .rejudge_case_ids(
        checkpoint_path=(
            CHECKPOINT_PATH
        ),
        case_ids=(
            BRAND_CASES
        ),
    )
)

display(
    rejudged[
        [
            "case_id",
            "split",
            "overall_judge_score",
            "correctness",
            "groundedness",
            "caveat_compliance",
            "completeness",
            "managerial_usefulness",
            "instruction_following",
            "judge_failure_tags",
            "judge_summary_reason",
        ]
    ]
)

a004: judge-only rerun score=3.10
a012: judge-only rerun score=5.00


,case_id,split,overall_judge_score,correctness,groundedness,caveat_compliance,completeness,managerial_usefulness,instruction_following,judge_failure_tags,judge_summary_reason
0,a004,dev,3.1,2.0,3.0,5.0,2.0,3.0,3.0,"[missed_requested_metric, semantic_numeric_mis...",پاسخ از نظر سیاست‌گذاری محتاط و درست است و سهم...
1,a012,test,5.0,5.0,5.0,5.0,5.0,5.0,5.0,[],پاسخ کاملاً منطبق با سیاست برند است: ادعای سهم...


In [4]:
results = evaluator.run(
    checkpoint_path=(
        CHECKPOINT_PATH
    ),
    splits=[
        "dev",
        "test",
    ],
    force=False,
)

summary = (
    summarize_analytics_results(
        results
    )
)

display(
    summary
)

display(
    analytics_failure_summary(
        results
    )
)

[1/15] a001: checkpoint
[2/15] a002: checkpoint
[3/15] a003: checkpoint
[4/15] a004: checkpoint
[5/15] a005: checkpoint
[6/15] a006: checkpoint
[7/15] a007: checkpoint
[8/15] a008: checkpoint
[9/15] a009: checkpoint
[10/15] a010: checkpoint
[11/15] a011: checkpoint
[12/15] a012: checkpoint
[13/15] a013: checkpoint
[14/15] a014: checkpoint
[15/15] a015: checkpoint


,split,cases,successful_cases,overall_judge_score,numeric_faithfulness,fact_value_accuracy,scope_product_count_accuracy,comparison_fact_accuracy,rendered_metric_accuracy,policy_guard_configuration,...,completeness,relevance,managerial_usefulness,instruction_following,answer_latency_ms,judge_latency_ms,evaluation_latency_ms,answer_total_tokens,judge_total_tokens,answer_latency_p95_ms
0,dev,5,5,4.620,1.0,1.0,1.0,1.0,1.0,1.0,...,4.4,4.800000,4.6,4.600000,10477.990995,9411.007516,20254.334624,3206.2,3105.200000,16385.477017
1,test,10,10,4.995,1.0,1.0,1.0,1.0,1.0,1.0,...,5.0,5.000000,4.9,5.000000,14493.414109,9863.255376,24837.247226,3537.7,3224.200000,20666.823143
2,ALL,15,15,4.870,1.0,1.0,1.0,1.0,1.0,1.0,...,4.8,4.933333,4.8,4.866667,13154.939738,9712.506089,23309.609692,3427.2,3184.533333,19771.165598


,failure,count
0,missed_requested_metric,1
1,semantic_numeric_misinterpretation,1
